# Decision-focused learning with quarterly PD and Basel IRB (empirical, FRED)

Empirical counterpart of `04_dfl_simulated_panel`: the DFL model of the methodology
chapter (S3.2.4) keeps the interpretable PD forecast (eq. 3.49) but trains it end-to-end
on decision regret (eq. 3.33) through the differentiable allocation layer (eq. 3.50).
The harness is copied from `test_blackbox_quarterly_pd_irb_v2` /
`test_predict_then_optimise_quarterly_pd_irb_v2` - same data loading, alignment, decision
environment, evaluation splits (`modern`, `stress`), metrics and plots - and the PTO
models plus the persistence baseline are **re-run here on identical expanding windows**.

**The allocation layer** (see the DFL core section): forward pass solves the smooth
forecast-based objective (eq. 3.24/3.25) by bisection on the first-order condition;
backward pass applies the closed-form implicit-function-theorem ratio
dalpha*/dpd = -(dF/dpd)/(dF/dalpha) at the optimum (zero at boundary optima). The
solver is never differentiated through. Full chain (eq. 3.52): realised regret
subgradient (eq. 3.43) x IFT ratio x network Jacobian.

Empirical specifics (matching the black-box notebook): the capital requirement in the
training loss is `k_irb(pd_a_realised_{t+1})`, matching how the empirical evaluation
computes realised utility; the foresight oracle is the only oracle; regret vs perfect
foresight is the decision metric. The Vasicek correlation in the decision objective is
the Basel R evaluated at the annualised forecast (held fixed when differentiating -
documented gradient simplification, quantified in the gradient-audit cell).

**Comparability caveat for the write-up**: the empirical PTO decision rule is the
plug-in grid rule (kinked utility at the point forecast), while DFL necessarily decides
through the smooth expected-utility objective - the smooth objective is what makes the
IFT gradient exist. DFL therefore differs from empirical PTO in BOTH the training loss
and the decision map. The simulated comparison (notebook 04 vs 02 with
`DECISION_RULE = "expected"`) isolates the training-loss effect cleanly; interpret the
empirical DFL-vs-PTO gap with that in mind.

In [ ]:
from __future__ import annotations

import time
from dataclasses import dataclass
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import norm
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler

try:
    import torch
    from torch import nn
    from torch.utils.data import DataLoader, TensorDataset
except ImportError as exc:
    raise ImportError(
        "This notebook needs PyTorch for the linear and MLP PTO models. "
        "Install the project requirements or select the project kernel."
    ) from exc

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "data" / "processed" / "fred_loan_return_aligned.csv").exists():
    ROOT = ROOT.parent

FIGURES_DIR = ROOT / "figures" / "dfl_pd_irb_v2"
RESULTS_DIR = ROOT / "models" / "notebooks" / "results"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f"Project root: {ROOT}")
print(f"Figures: {FIGURES_DIR.relative_to(ROOT)}")
print(f"Results: {RESULTS_DIR.relative_to(ROOT)}")

## Configuration

`CHARGE_OFF_CONVERSION = "simple"` inverts FRED's simple x4 annualisation, which is the
correct inverse for these series; `"compound"` remains available if the source were treated
as a compounded annual rate (the numerical difference at these PD levels is under 1%
relative). `TARGET_TRANSFORM = "logit"` trains on logit(PD) with a raw-score output head;
set `"level"` to reproduce the original sigmoid + level-space MSE behaviour.
`MLP_SEEDS` controls the ensemble size; MLP runtime scales linearly in the number of seeds.
The `stress` split retrains from 2006Q1 onward and roughly doubles total runtime — drop it
from `EVAL_SPLITS` for quick iterations.
The `LOSS` / `NLL_*` / `TARGET_TRANSFORM` switches govern the **PTO predictor only**,
re-run here unchanged for the aligned comparison (`NLL_RHO_MODE = "auto"` stays the
empirical default). DFL keeps the PD intermediate but trains it on realised regret through the implicit
allocation layer; the prediction-loss switches never touch it. `DFL_SEEDS` / `DFL_MODEL_CFG` pin the
DFL model to the same seed-ensemble effort and capacity/optimiser/early-stopping budget
as the PTO MLP (fairness rule). Runtime roughly doubles versus the PTO notebook because
the DFL linear + MLP ensemble train at every window as well.


In [ ]:
# Data conventions
CHARGE_OFF_IS_PERCENT = True
# FRED annualises quarterly net charge-offs with a simple x4, so "simple" (/4) is the
# correct inverse of the source convention. "compound" kept as an alternative.
CHARGE_OFF_CONVERSION = "simple"
HORIZON = 1
DATE_COL = "DATE"
TARGET_COL = "charge_off_rate"

# Feature policy: the current-quarter charge-off c_q is realised by decision time
# (start of t+1), modulo a reporting lag. Set False to revert to lag-1 only.
INCLUDE_CURRENT_CQ = True

# Decision and Basel constants
LGD = 0.45
M = 2.5
K1 = 1.0
LEVERAGE = 10.0
Y_BAR = 0.0168
LAMBDA = 10.0

# Basel corporate PD floor (0.03%), applied inside k_irb_from_annual_pd.
APPLY_PD_FLOOR = True
PD_FLOOR = 3e-4

PD_Q_CLIP_LOW = 1e-6
PD_Q_CLIP_HIGH = 0.99
PD_A_CLIP_LOW = 1e-6
PD_A_CLIP_HIGH = 0.99
ALPHA_GRID = np.linspace(0.0, 1.0, 101)

# Modelling choices
TARGET_TRANSFORM = "logit"  # "logit" or "level" (original behaviour)
MLP_SEEDS = (42, 43, 44, 45, 46)  # MLP seed ensemble; use (42,) to disable

# Training loss. "nll_vasicek" fits PD by maximum likelihood under the assumption that
# realised charge-off rates are Vasicek draws given PD, with correlation = Basel R times
# NLL_RHO_SCALE. Unlike logit-space MSE (which targets the conditional MEDIAN of the
# realisation), the NLL targets the MEAN PD. Caveat: the aggregate FRED series,
# conditioned on rich macro features, carries far less residual noise than full Basel
# rho implies, so NLL_RHO_SCALE = 1.0 may inflate small PDs (roughly by the inverse
# median map, ~2-2.5x at rho ~ 0.2). Reduce NLL_RHO_SCALE or set LOSS = "mse" to compare.
LOSS = "nll_vasicek"  # "nll_vasicek" or "mse"
# "auto" estimates the residual probit-space variance per training window (OLS, no
# look-ahead) and sets rho_eff = s2 / (1 + s2): the likelihood noise the data actually
# exhibits given the features. Recommended for the aggregate FRED series, where full
# Basel rho drastically overstates residual noise and inflates predictions ~2-2.5x.
# "basel" uses rho = NLL_RHO_SCALE * BaselR(pd_a_hat).
NLL_RHO_MODE = "auto"  # "auto" or "basel"
NLL_RHO_SCALE = 1.0    # only used when NLL_RHO_MODE == "basel"

# Evaluation splits: separate expanding-window experiments.
EVAL_SPLITS = {
    "modern": {"initial_train_frac": 0.75},
    "stress": {"first_test_date": "2006-01-01"},
}

@dataclass
class ModelConfig:
    hidden_dim: int = 16
    lr: float = 1e-3
    weight_decay: float = 1e-4
    batch_size: int = 16
    epochs: int = 500
    patience: int = 75
    val_frac: float = 0.2
    seed: int = 42

MODEL_CFG = ModelConfig()

# ---- DFL additions ---------------------------------------------------------------------
# Decision-focused learning keeps the PD forecaster and trains it on realised regret
# through the implicit allocation layer (smooth objective + bisection forward, closed-form
# IFT backward; see the DFL core section). The LOSS / NLL_* / TARGET_TRANSFORM switches
# above govern the PTO predictor ONLY and never touch DFL.
DFL_SEEDS = MLP_SEEDS           # same seed-ensemble effort as the PTO MLP (fairness rule)
DFL_MODEL_CFG = MODEL_CFG       # identical capacity / optimiser / early-stopping budget
DFL_ALPHA_LO = 1e-4             # lower bisection bracket (alpha* = 0 handled as boundary)
DFL_BISECT_ITERS = 60           # bracket halves 60x: alpha* to ~1e-18, far below grid step
DFL_FD_REL = 1e-5               # relative step for the K'_IRB central finite difference
DFL_GRAD_CLIP = 1e3             # safety clamp on |dalpha*/dpd| (thresholds near support edge)
BVN_GL_NODES = 24               # Gauss-Legendre nodes for the bivariate normal CDF
print(f"DFL: PD forecaster trained on realised regret through the implicit allocation layer "
      f"(bisection + IFT); seeds {DFL_SEEDS}; same ModelConfig as PTO; rho held fixed in the gradient.")

## Assumptions and caveats

These are held fixed by design; each materially shapes the results and should be restated
wherever results are reported.

- **Constant LGD = 0.45.** Realised PD is backed out as `c_q / LGD`, so "PD" here is a
  rescaled charge-off rate. LGD is procyclical in reality and Basel requires downturn LGD.
- **Corporate IRB curve on an aggregate charge-off series.** The asset-correlation function,
  maturity adjustment, and 0.03% floor are the wholesale/corporate ones. If the underlying
  FRED series is retail/consumer, the retail risk-weight functions (different correlation,
  no maturity adjustment) would apply.
- **Point-in-time PD in a through-the-cycle formula.** Next-quarter PD is annualised as
  `1-(1-pd_q)^4` (independence across quarters) and fed into the IRB formula, making capital
  procyclical; Basel intends TTC PDs.
- **Constant quarterly yield `Y_BAR` and constant leverage** regardless of the rate
  environment.
- **Final-vintage macro data.** Features use revised series, and quarter-t values (e.g. GDP)
  are not published until well into t+1 — a mild look-ahead relative to real-time
  deployment. Including `c_q` as a feature likewise assumes the quarter-t charge-off is
  observable at decision time; set `INCLUDE_CURRENT_CQ = False` to enforce a full
  one-quarter reporting lag.
- **Oracle regret** uses per-period perfect foresight of both `c_q` and the annualised PD
  entering the capital requirement, so regret mixes forecast error with capital-requirement
  foresight.
- **Capital buffer add-on.** Committed capital is excluded from the available buffer;
  equivalently the requirement is `K_IRB + 1/LEVERAGE` per unit exposure, i.e. the
  regulatory minimum plus a fixed management buffer. Calibrated so the constraint binds
  in the interior of the allocation range - without it the constraint never binds for
  alpha in [0,1] and the decision is degenerate at alpha = 1.
- **DFL training signal is hindsight-outcome information only**: per training sample it
  uses realised `c_q_{t+1}`, the realised annualised PD's IRB charge
  `k_irb(pd_a_realised_{t+1})`, and the foresight-oracle utility - the identical
  training signal to the black-box benchmark, and the same information status as PTO
  training on realised `c_{t+1}/LGD`. Targets only; features identical across models.
- **DFL decides through the smooth expected-utility objective, solved continuously**
  (bisection on the FOC; grid snap at evaluation only). The empirical PTO decision rule
  remains the plug-in grid rule, so empirical DFL-vs-PTO differences mix the training
  loss and the decision map - see the header caveat; the simulated notebooks isolate
  the training-loss effect.
- **Correlation channel omitted from the gradient**: rho = BaselR(pd_a_hat) is held
  fixed at its evaluated value when differentiating (forward pass exact; audited below).
  K'_IRB uses a central finite difference on the composite quarterly -> annualised ->
  Basel map (annualisation chain 4(1-pd)^3 included automatically).

## Quarterly PD, Basel IRB, and allocation objective

In [ ]:
def annual_chargeoff_to_quarterly(c_annual, conversion=CHARGE_OFF_CONVERSION):
    c_annual = np.asarray(c_annual, dtype=float)
    if conversion == "compound":
        return 1.0 - np.power(1.0 - c_annual, 1.0 / 4.0)
    if conversion == "simple":
        return c_annual / 4.0
    raise ValueError("CHARGE_OFF_CONVERSION must be 'compound' or 'simple'.")


def clip_pd_q(pd_q):
    return np.clip(np.asarray(pd_q, dtype=float), PD_Q_CLIP_LOW, PD_Q_CLIP_HIGH)


def clip_pd_a(pd_a):
    return np.clip(np.asarray(pd_a, dtype=float), PD_A_CLIP_LOW, PD_A_CLIP_HIGH)


def quarterly_pd_to_annual(pd_q):
    pd_q = clip_pd_q(pd_q)
    return clip_pd_a(1.0 - np.power(1.0 - pd_q, 4.0))


def k_irb_from_annual_pd(pd_a, lgd=LGD, m=M, apply_floor=APPLY_PD_FLOOR):
    # Basel IRB capital requirement per unit exposure. Input must be annual PD.
    pd_a = clip_pd_a(pd_a)
    if apply_floor:
        pd_a = np.maximum(pd_a, PD_FLOOR)
    exp_term = (1.0 - np.exp(-50.0 * pd_a)) / (1.0 - np.exp(-50.0))
    r = 0.12 * exp_term + 0.24 * (1.0 - exp_term)
    b = np.square(0.11852 - 0.05478 * np.log(pd_a))
    maturity_adj = (1.0 + (m - 2.5) * b) / (1.0 - 1.5 * b)
    z = norm.ppf(pd_a) / np.sqrt(1.0 - r) + np.sqrt(r / (1.0 - r)) * norm.ppf(0.999)
    k_irb = lgd * (norm.cdf(z) - pd_a) * maturity_adj
    return np.maximum(k_irb, 0.0)


def allocation_utility(alpha, c_q, pd_a, leverage=LEVERAGE, y_bar=Y_BAR, lambda_penalty=LAMBDA, k1=K1, lgd=LGD, m=M):
    # U(alpha; c_q, pd_a), with pd_a used only for Basel IRB capital.
    # Buffer semantics (intentional): committed capital alpha*k1 is excluded from the
    # available buffer. Algebraically identical to requiring K_IRB plus a fixed
    # management buffer add-on of 1/LEVERAGE per unit exposure; this is what makes the
    # capital constraint bind in the interior of the alpha grid (without it the
    # constraint never binds on [0,1] and every model would choose alpha = 1).
    alpha = np.asarray(alpha, dtype=float)
    c_q = float(c_q)
    pd_a = float(clip_pd_a(pd_a))
    exposure = leverage * alpha * k1
    terminal_capital = k1 + exposure * (y_bar - c_q)
    committed_capital = alpha * k1
    available_buffer = terminal_capital - committed_capital
    required_capital = k_irb_from_annual_pd(pd_a, lgd=lgd, m=m) * exposure
    shortfall = np.maximum(required_capital - available_buffer, 0.0)
    return terminal_capital - lambda_penalty * shortfall


def decision_components(alpha, c_q, pd_a):
    exposure = LEVERAGE * alpha * K1
    terminal_capital = K1 + exposure * (Y_BAR - c_q)
    committed_capital = alpha * K1
    available_buffer = terminal_capital - committed_capital
    k_irb = float(k_irb_from_annual_pd(pd_a))
    required_capital = k_irb * exposure
    shortfall = max(required_capital - available_buffer, 0.0)
    utility = terminal_capital - LAMBDA * shortfall
    return dict(exposure=exposure, terminal_capital=terminal_capital, committed_capital=committed_capital,
                available_buffer=available_buffer, required_capital=required_capital, shortfall=shortfall,
                utility=utility, k_irb=k_irb)


def optimise_alpha_grid(c_q, pd_a, alpha_grid=ALPHA_GRID):
    utilities = allocation_utility(alpha_grid, c_q=c_q, pd_a=pd_a)
    best_idx = int(np.argmax(utilities))
    return float(alpha_grid[best_idx]), float(utilities[best_idx])

## Load and align data

The forecasting row is `x_t -> PD_Q_{t+1}`; result columns are named for the realised
next-period outcome, because that is the period used for decision evaluation.

Features now include the current-quarter charge-off `c_q` (realised by decision time,
subject to a reporting lag — see the assumptions cell) alongside `c_q_lag1`. Constructed PD
columns (`pd_q`, `pd_a`) remain excluded: realised PD is built from charge-offs for the
target/evaluation convention, not treated as an observed signal.
After loading, the per-quarter foresight oracle (`alpha_oracle_next`,
`utility_oracle_next`) and the realised IRB charge `k_irb_realised_next` are precomputed
once: they are the DFL model's training signal and are identical to the quantities
`build_result_row` recomputes per test row.


In [ ]:
def load_quarterly_pd_panel(macro_csv, chargeoff_csv, date_col=DATE_COL, target_col=TARGET_COL, feature_cols=None, horizon=HORIZON):
    macro = pd.read_csv(macro_csv)
    charge = pd.read_csv(chargeoff_csv)
    macro[date_col] = pd.to_datetime(macro[date_col]).dt.to_period("Q").dt.start_time
    charge[date_col] = pd.to_datetime(charge[date_col]).dt.to_period("Q").dt.start_time

    if target_col not in charge.columns:
        raise ValueError(f"{target_col!r} not found in charge-off columns: {list(charge.columns)}")

    charge["c_annual_raw"] = pd.to_numeric(charge[target_col], errors="coerce")
    charge["c_annual_decimal"] = charge["c_annual_raw"] / 100.0 if CHARGE_OFF_IS_PERCENT else charge["c_annual_raw"]
    charge["c_q"] = annual_chargeoff_to_quarterly(charge["c_annual_decimal"].to_numpy())
    charge["pd_q"] = clip_pd_q(charge["c_q"].to_numpy() / LGD)
    charge["pd_a"] = quarterly_pd_to_annual(charge["pd_q"].to_numpy())

    keep_charge = [date_col, "c_annual_raw", "c_annual_decimal", "c_q", "pd_q", "pd_a"]
    df = pd.merge(macro, charge[keep_charge], on=date_col, how="inner").sort_values(date_col).reset_index(drop=True)

    # Observed autoregressive signals. PD columns are never features.
    df["c_q_lag1"] = df["c_q"].shift(1)

    df["target_date"] = df[date_col].shift(-horizon)
    df["c_annual_raw_next"] = df["c_annual_raw"].shift(-horizon)
    df["c_q_realised"] = df["c_q"].shift(-horizon)
    df["pd_q_realised"] = df["pd_q"].shift(-horizon)
    df["pd_a_realised"] = df["pd_a"].shift(-horizon)
    df["target_pd_q_next"] = df["pd_q_realised"]

    if feature_cols is None:
        exclude = {date_col, "c_annual_raw", "c_annual_decimal", "pd_q", "pd_a", "target_date",
                   "c_annual_raw_next", "c_q_realised", "pd_q_realised", "pd_a_realised", "target_pd_q_next"}
        if not INCLUDE_CURRENT_CQ:
            exclude.add("c_q")
        feature_cols = [c for c in df.columns if c not in exclude]
    else:
        feature_cols = list(feature_cols)

    for col in feature_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    required = feature_cols + ["c_q", "target_date", "c_annual_raw_next", "c_q_realised", "pd_q_realised", "pd_a_realised", "target_pd_q_next"]
    required = list(dict.fromkeys(required))
    df = df.dropna(subset=required).reset_index(drop=True)
    if len(df) < 30:
        raise ValueError(f"Only {len(df)} usable observations after alignment; check dates and missing values.")
    df.attrs["feature_cols"] = feature_cols
    return df

macro_csv = ROOT / "data" / "processed" / "macro_panel_quarterly.csv"
chargeoff_csv = ROOT / "data" / "processed" / "fred_loan_return_aligned.csv"
df = load_quarterly_pd_panel(macro_csv, chargeoff_csv)
feature_cols = df.attrs["feature_cols"]

print(f"Observations after alignment and one-step target: {len(df)}")
print(f"Feature date range: {df[DATE_COL].min().date()} to {df[DATE_COL].max().date()}")
print(f"Target date range: {df['target_date'].min().date()} to {df['target_date'].max().date()}")
print(f"Features ({len(feature_cols)}): {feature_cols}")
assert "c_q_lag1" in feature_cols
assert ("c_q" in feature_cols) == INCLUDE_CURRENT_CQ
assert not any(col.startswith("pd_q") or col.startswith("pd_a") for col in feature_cols)
print(f"Lag features included: c_q_lag1{' and current-quarter c_q' if INCLUDE_CURRENT_CQ else ''}; PD columns excluded.")
display(df.head())

# Foresight-oracle allocation and utility per TARGET quarter (perfect hindsight of
# c_q_{t+1} and pd_a_realised_{t+1}; identical to build_result_row's recomputation).
# Hindsight-outcome information: used only as the DFL model's training signal, never as
# a feature.
_oracle = [optimise_alpha_grid(c_q=float(c), pd_a=float(p))
           for c, p in zip(df["c_q_realised"], df["pd_a_realised"])]
df["alpha_oracle_next"] = [a for a, _ in _oracle]
df["utility_oracle_next"] = [u for _, u in _oracle]
# Capital requirement entering the DFL training loss: the realised annualised PD's
# IRB charge, matching how the empirical evaluation computes realised utility.
df["k_irb_realised_next"] = k_irb_from_annual_pd(df["pd_a_realised"].to_numpy())
assert not set(["alpha_oracle_next", "utility_oracle_next", "k_irb_realised_next"]) & set(feature_cols)
print("Precomputed foresight oracle and realised K_IRB for the DFL training signal.")


## PTO models and training loop (verbatim from the PTO notebook)

Both models output a raw score whose sigmoid is the predicted quarterly PD. The loss
is selected by `LOSS`: Vasicek negative log-likelihood on probit targets (default;
targets the mean PD under the assumed Vasicek realisation noise) or MSE on
logit/level targets. Under `LOSS = "mse"`,
`TARGET_TRANSFORM` picks logit-space targets (well-conditioned) or level-space
targets (the original objective).
The output bias is initialised at logit(mean PD) of the pre-validation slice only.
Mini-batches are shuffled with a seeded generator; the train/validation split itself
remains chronological.

In [ ]:
def logit_np(p):
    p = np.clip(np.asarray(p, dtype=float), PD_Q_CLIP_LOW, 1.0 - PD_Q_CLIP_LOW)
    return np.log(p / (1.0 - p))


def probit_np(p):
    p = np.clip(np.asarray(p, dtype=float), PD_Q_CLIP_LOW, 1.0 - PD_Q_CLIP_LOW)
    return norm.ppf(p)


def torch_vasicek_nll(score, y_probit, rho_fixed=None):
    """Negative log-likelihood of realised rates under the Vasicek density,
    parameterised by the model's predicted PD (sigmoid of the raw score).

    In probit space the Vasicek model is Gaussian:
        ndtri(x) | p ~ N( ndtri(p) / sqrt(1 - rho), rho / (1 - rho) ),
    with rho = NLL_RHO_SCALE * BaselR(annualised p). Terms not involving the
    prediction are dropped, except the log-variance, which depends on p through rho.
    Maximising this likelihood targets the MEAN parameter p.
    """
    pd_q = torch.sigmoid(score).clamp(PD_Q_CLIP_LOW, PD_Q_CLIP_HIGH)
    if rho_fixed is not None:
        rho = torch.as_tensor(float(rho_fixed))
    else:
        pd_a = (1.0 - torch.pow(1.0 - pd_q, 4.0)).clamp(PD_A_CLIP_LOW, PD_A_CLIP_HIGH)
        exp_term = (1.0 - torch.exp(-50.0 * pd_a)) / (1.0 - float(np.exp(-50.0)))
        rho = 0.12 * exp_term + 0.24 * (1.0 - exp_term)
        rho = torch.clamp(NLL_RHO_SCALE * rho, 1e-4, 0.999)
    mu = torch.special.ndtri(pd_q) / torch.sqrt(1.0 - rho)
    var = rho / (1.0 - rho)
    return torch.mean(0.5 * torch.log(var) + (y_probit - mu) ** 2 / (2.0 * var))


class PDForecaster(nn.Module):
    # Outputs a raw score; predicted quarterly PD is sigmoid(score).
    def __init__(self, input_dim, model_type, hidden_dim=16):
        super().__init__()
        if model_type == "linear":
            self.net = nn.Linear(input_dim, 1)
        elif model_type == "mlp":
            self.net = nn.Sequential(nn.Linear(input_dim, hidden_dim), nn.ReLU(),
                                     nn.Linear(hidden_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, 1))
        else:
            raise ValueError("model_type must be 'linear' or 'mlp'.")

    def forward(self, x):
        return self.net(x).squeeze(-1)


def init_output_bias_for_target(model, y_pre_val):
    # Bias starts at logit(mean PD) of the PRE-validation slice only, so validation
    # targets used for early stopping never influence initialisation.
    y_mean = float(np.clip(np.mean(y_pre_val), PD_Q_CLIP_LOW, 1.0 - PD_Q_CLIP_LOW))
    logit_mean = float(np.log(y_mean / (1.0 - y_mean)))
    with torch.no_grad():
        if isinstance(model.net, nn.Linear):
            model.net.bias.fill_(logit_mean)
        else:
            model.net[-1].bias.fill_(logit_mean)


def train_pd_model(x_train, y_train_pd_q, model_type, model_cfg, seed=None, return_history=False):
    seed = model_cfg.seed if seed is None else int(seed)
    torch.manual_seed(seed)
    np.random.seed(seed)
    n = x_train.shape[0]
    val_size = max(1, int(np.floor(model_cfg.val_frac * n)))
    train_size = n - val_size
    if train_size < 8:
        raise ValueError("Training set too small after validation split.")

    x_tr, x_val = x_train[:train_size], x_train[train_size:]
    y_tr, y_val = y_train_pd_q[:train_size], y_train_pd_q[train_size:]
    if LOSS == "nll_vasicek":
        t_tr, t_val = probit_np(y_tr), probit_np(y_val)
    elif LOSS == "mse" and TARGET_TRANSFORM == "logit":
        t_tr, t_val = logit_np(y_tr), logit_np(y_val)
    elif LOSS == "mse" and TARGET_TRANSFORM == "level":
        t_tr, t_val = y_tr, y_val
    else:
        raise ValueError("LOSS must be 'nll_vasicek' or 'mse'; TARGET_TRANSFORM must be 'logit' or 'level'.")

    train_ds = TensorDataset(torch.tensor(x_tr, dtype=torch.float32), torch.tensor(t_tr, dtype=torch.float32))
    loader_gen = torch.Generator().manual_seed(seed)
    train_loader = DataLoader(train_ds, batch_size=model_cfg.batch_size, shuffle=True, generator=loader_gen)

    model = PDForecaster(input_dim=x_train.shape[1], model_type=model_type, hidden_dim=model_cfg.hidden_dim)
    init_output_bias_for_target(model, y_tr)
    optimiser = torch.optim.AdamW(model.parameters(), lr=model_cfg.lr, weight_decay=model_cfg.weight_decay)
    mse_fn = nn.MSELoss()

    rho_fixed = None
    if LOSS == "nll_vasicek" and NLL_RHO_MODE == "auto":
        # Residual probit-space variance of the training window (pre-validation slice),
        # via OLS with intercept: rho_eff = s2 / (1 + s2) is the Vasicek correlation
        # implied by the noise the data actually shows given the features.
        X_ols = np.hstack([x_tr, np.ones((len(x_tr), 1))])
        beta_ols, *_ = np.linalg.lstsq(X_ols, t_tr, rcond=None)
        resid = t_tr - X_ols @ beta_ols
        dof = max(len(t_tr) - X_ols.shape[1], 1)
        s2 = float(resid @ resid) / dof
        rho_fixed = float(np.clip(s2 / (1.0 + s2), 1e-4, 0.5))

    def compute_loss(score, target):
        if LOSS == "nll_vasicek":
            return torch_vasicek_nll(score, target, rho_fixed=rho_fixed)
        pred = score if TARGET_TRANSFORM == "logit" else torch.sigmoid(score)
        return mse_fn(pred, target)

    x_val_t = torch.tensor(x_val, dtype=torch.float32)
    t_val_t = torch.tensor(t_val, dtype=torch.float32)
    best_state, best_val, stale_epochs, history = None, float("inf"), 0, []

    for epoch in range(model_cfg.epochs):
        model.train()
        batch_losses = []
        for xb, tb in train_loader:
            optimiser.zero_grad()
            loss = compute_loss(model(xb), tb)
            loss.backward()
            optimiser.step()
            batch_losses.append(loss.item())

        model.eval()
        with torch.no_grad():
            score_val = model(x_val_t)
            val_loss = compute_loss(score_val, t_val_t).item()
            pd_q_val_np = clip_pd_q(torch.sigmoid(score_val).cpu().numpy())
        history.append({"epoch": epoch + 1, "model": model_type, "seed": seed,
                        "rho_eff": float("nan") if rho_fixed is None else rho_fixed,
                        "train_loss": float(np.mean(batch_losses)), "val_loss": float(val_loss),
                        "val_rmse_pd_q": float(np.sqrt(mean_squared_error(y_val, pd_q_val_np))),
                        "val_mae_pd_q": float(mean_absolute_error(y_val, pd_q_val_np)),
                        "val_mean_pd_q_hat": float(np.mean(pd_q_val_np))})

        if val_loss < best_val - 1e-12:
            best_val, stale_epochs = val_loss, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            stale_epochs += 1
        if stale_epochs >= model_cfg.patience:
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    return (model, pd.DataFrame(history)) if return_history else model


def predict_pd_q(model, x):
    model.eval()
    with torch.no_grad():
        pred = torch.sigmoid(model(torch.tensor(x, dtype=torch.float32))).cpu().numpy()
    return clip_pd_q(np.atleast_1d(pred))


def train_mlp_ensemble(x_train, y_train_pd_q, model_cfg, seeds=MLP_SEEDS, return_history=False):
    models, first_hist = [], None
    for i, s in enumerate(seeds):
        want_hist = return_history and i == 0
        out = train_pd_model(x_train, y_train_pd_q, "mlp", model_cfg, seed=s, return_history=want_hist)
        if want_hist:
            model, first_hist = out
        else:
            model = out
        models.append(model)
    return models, first_hist


def predict_pd_q_ensemble(models, x):
    preds = np.stack([predict_pd_q(m, x) for m in models], axis=0)
    return preds.mean(axis=0), preds.std(axis=0)

## DFL core: smooth objective, bisection forward pass, IFT backward pass

The cell below implements the differentiable allocation layer:

- `dfl_smooth_utility` / `dfl_foc`: the forecast-based objective U_hat(alpha; pd) (thesis
  eq. 3.24) and its first-order condition F = dU_hat/dalpha, in closed form via the
  Vasicek expected shortfall ES = Phi2(ndtri(pd), y_x; sqrt(rho)) - x Phi(y_x)
  (eq. 3.26-3.28; `bvn_cdf` is a vectorised Drezner-Wesolowsky bivariate normal CDF);
- `solve_alpha_star`: vectorised bisection on F = 0. F is strictly decreasing wherever
  the Vasicek density at the threshold is positive (d2U/dalpha2 =
  -lambda f_L(x) / (ell alpha^3 LGD) < 0), so endpoint sign checks detect boundary
  optima (F(1) >= 0 -> alpha* = 1; F(lo) <= 0 -> alpha* = 0) and bisection converges to
  the unique interior root otherwise. The same call returns the closed-form IFT
  gradient dalpha*/dpd (zero at boundaries), assembled from: the threshold channel
  via K'_IRB (central finite difference on the composite quarterly -> annualised ->
  Basel map, annualisation chain 4(1-pd)^3 included), the distribution-mean channel
  via Psi = Phi((y_x - sqrt(rho) ndtri(pd)) / sqrt(1-rho)) and the exceedance-density
  term, with rho held fixed (documented simplification);
- `ImplicitAllocation`: the torch autograd wrapper - non-differentiable bisection in
  forward, the IFT ratio in backward;
- `train_dfl_model` / `train_dfl_ensemble`: identical mechanics to the black-box
  training loop (chronological validation split, shuffled mini-batches, early stopping
  on validation regret, same ModelConfig and seed ensemble), with the network output
  read as a PD forecast and passed through the implicit layer;
- `snap_alpha`: evaluation-only grid snap (training stays continuous - snapping inside
  training would reintroduce the zero-gradient step function).

In [ ]:
def torch_allocation_utility(alpha, c_q, k):
    """Differentiable realised utility - same constants and the same intentional buffer
    semantics as allocation_utility (committed capital alpha*K1 excluded from the buffer).
    The kinked shortfall max(., 0) is piecewise linear; torch subgradients handle it
    (thesis eq. 3.43: the realised regret subgradient shared by BB and DFL)."""
    exposure = LEVERAGE * alpha * K1
    terminal_capital = K1 + exposure * (Y_BAR - c_q)
    committed_capital = alpha * K1
    available_buffer = terminal_capital - committed_capital
    required_capital = k * exposure
    shortfall = torch.clamp(required_capital - available_buffer, min=0.0)
    return terminal_capital - LAMBDA * shortfall


def snap_alpha(alpha):
    """Snap continuous outputs to the shared decision grid at EVALUATION time only, so
    every model acts in the identical environment (foresight regret >= 0 exactly).
    Training uses the continuous bisection optimum - snapping inside training would
    reintroduce the zero-gradient step function the IFT exists to avoid."""
    alpha = np.atleast_1d(np.asarray(alpha, dtype=float))
    idx = np.abs(ALPHA_GRID[None, :] - alpha[:, None]).argmin(axis=1)
    return ALPHA_GRID[idx]


# ---------------------------------------------------------------------------------------
# DFL core: smooth forecast-based objective, bisection forward pass, IFT backward pass.
#
# Forward decision map (thesis eq. 3.24/3.25): given the model's own quarterly PD forecast,
#   alpha*(pd_q) = argmax_alpha  1 + ell*alpha*(y_bar - LGD*pd_q)
#                              - lambda*ell*alpha*LGD * E[(L - x(alpha, pd_q))^+]
# with L ~ Vasicek(pd_q, rho), rho = DFL_RHO_SCALE * BaselR(pd_a(pd_q)) (matching the
# simulator / expected-utility PTO convention), and threshold
#   x(alpha, pd_q) = ((1 - alpha) + ell*alpha*(y_bar - k)) / (ell*alpha*LGD),
#   k = K_IRB(pd_a(pd_q)).
# The optimum solves the FOC F(alpha, pd_q) = 0 with
#   F = ell*(y_bar - LGD*pd_q) - lambda*ell*LGD*ES(x; pd_q, rho) - (lambda/alpha)*P(L > x).
# F is strictly decreasing where the Vasicek density at the threshold is positive
# (d2U/dalpha2 = -lambda * f_L(x) / (ell * alpha^3 * LGD) < 0), so a sign check at the
# brackets detects boundary solutions and bisection finds the unique interior root.
#
# Backward pass (implicit function theorem on the FOC):
#   dalpha*/dpd = - (dF/dpd) / (dF/dalpha) |_(alpha*)     [zero at boundary solutions]
#   dF/dalpha = -lambda * f_L(x) / (ell * alpha^3 * LGD)
#   dF/dpd    = -ell*LGD - lambda*ell*LGD*( P(L>x)*Kp/LGD + Psi )
#               - (lambda/alpha)*( f_L(x)*Kp/LGD + phi(y_x)/(sqrt(rho)*phi(ndtri(pd))) )
# where Kp = d K_IRB(pd_a(pd_q)) / d pd_q (central finite difference on the composite
# quarterly -> annual -> Basel map), and
#   Psi = Phi( (y_x - sqrt(rho)*ndtri(pd)) / sqrt(1-rho) )  in (0,1)
# is dES/dpd at fixed threshold (the y_x-channel terms cancel identically).
# Simplification (documented in the thesis): rho is held fixed at its evaluated value
# when differentiating; the correlation channel R'(pd) is omitted from the gradient.
# The forward pass always evaluates the exact rho.
# ---------------------------------------------------------------------------------------
from scipy.special import ndtri as _ndtri

DFL_RHO_SCALE = float(globals().get("RHO_SCALE", 1.0))
_BVN_GL_X, _BVN_GL_W = np.polynomial.legendre.leggauss(BVN_GL_NODES)
_X_EPS = 1e-9          # threshold considered outside (0,1) beyond this
_PD_LO, _PD_HI = PD_Q_CLIP_LOW, PD_Q_CLIP_HIGH


def dfl_basel_r_annual(pd_a):
    pd_a = clip_pd_a(pd_a)
    exp_term = (1.0 - np.exp(-50.0 * pd_a)) / (1.0 - np.exp(-50.0))
    return 0.12 * exp_term + 0.24 * (1.0 - exp_term)


def dfl_rho(pd_a):
    return np.clip(DFL_RHO_SCALE * dfl_basel_r_annual(pd_a), 1e-4, 0.999)


def bvn_cdf(a, b, r):
    """Bivariate standard normal CDF Phi2(a, b; r), vectorised, elementwise r.
    Drezner-Wesolowsky correlation-integral form with fixed Gauss-Legendre nodes:
    Phi2 = Phi(a)Phi(b) + (1/2pi) * int_0^r exp(-(a^2 - 2t a b + b^2)/(2(1-t^2))) / sqrt(1-t^2) dt.
    Accurate to ~1e-10 for |r| <= 0.95 (Basel sqrt(rho) is ~0.35-0.5)."""
    a = np.atleast_1d(np.asarray(a, dtype=float))
    b = np.atleast_1d(np.asarray(b, dtype=float))
    r = np.atleast_1d(np.asarray(r, dtype=float))
    t = 0.5 * r[:, None] * (_BVN_GL_X[None, :] + 1.0)          # nodes on [0, r]
    w = 0.5 * r[:, None] * _BVN_GL_W[None, :]
    one_m_t2 = 1.0 - t ** 2
    integrand = np.exp(-(a[:, None] ** 2 - 2.0 * t * a[:, None] * b[:, None] + b[:, None] ** 2)
                       / (2.0 * one_m_t2)) / np.sqrt(one_m_t2)
    return norm.cdf(a) * norm.cdf(b) + (integrand * w).sum(axis=1) / (2.0 * np.pi)


def dfl_threshold_x(alpha, k):
    return ((1.0 - alpha) + LEVERAGE * alpha * (Y_BAR - k)) / (LEVERAGE * alpha * LGD)


def vasicek_es_exceed(x, pd_q, rho):
    """Closed-form E[(L-x)^+] and P(L > x) under Vasicek(pd_q, rho); handles thresholds
    outside (0,1): x <= 0 -> hinge always active (ES = pd - x, P = 1); x >= 1 -> never
    active (ES = 0, P = 0). Returns (ES, P, y_x, inside_mask)."""
    x = np.atleast_1d(np.asarray(x, dtype=float))
    pd_q = np.atleast_1d(np.asarray(pd_q, dtype=float))
    rho = np.atleast_1d(np.asarray(rho, dtype=float))
    a = _ndtri(np.clip(pd_q, _PD_LO, 1.0 - _PD_LO))
    inside = (x > _X_EPS) & (x < 1.0 - _X_EPS)
    x_in = np.clip(x, _X_EPS, 1.0 - _X_EPS)
    y_x = (a - np.sqrt(1.0 - rho) * _ndtri(x_in)) / np.sqrt(rho)
    p_in = norm.cdf(y_x)
    es_in = bvn_cdf(a, y_x, np.sqrt(rho)) - x_in * p_in
    es = np.where(inside, es_in, np.where(x <= _X_EPS, pd_q - x, 0.0))
    p = np.where(inside, p_in, np.where(x <= _X_EPS, 1.0, 0.0))
    return es, p, y_x, inside


def vasicek_pdf(x, pd_q, rho):
    """Vasicek default-rate density f_L(x; pd_q, rho) (thesis eq. 3.7); 0 outside (0,1)."""
    x = np.atleast_1d(np.asarray(x, dtype=float))
    pd_q = np.atleast_1d(np.asarray(pd_q, dtype=float))
    rho = np.atleast_1d(np.asarray(rho, dtype=float))
    inside = (x > _X_EPS) & (x < 1.0 - _X_EPS)
    x_in = np.clip(x, _X_EPS, 1.0 - _X_EPS)
    a = _ndtri(np.clip(pd_q, _PD_LO, 1.0 - _PD_LO))
    xi = _ndtri(x_in)
    dens = (np.sqrt((1.0 - rho) / rho)
            * np.exp(-((np.sqrt(1.0 - rho) * xi - a) ** 2) / (2.0 * rho) + 0.5 * xi ** 2))
    return np.where(inside, dens, 0.0)


def dfl_smooth_utility(alpha, pd_q, k, rho):
    """Forecast-based expected utility U_hat(alpha; pd_q) (thesis eq. 3.24), closed form."""
    x = dfl_threshold_x(alpha, k)
    es, _, _, _ = vasicek_es_exceed(x, pd_q, rho)
    return (K1 + LEVERAGE * alpha * (Y_BAR - LGD * pd_q)
            - LAMBDA * LEVERAGE * alpha * LGD * es)


def dfl_foc(alpha, pd_q, k, rho):
    """F(alpha, pd_q) = dU_hat/dalpha."""
    x = dfl_threshold_x(alpha, k)
    es, p, _, _ = vasicek_es_exceed(x, pd_q, rho)
    return (LEVERAGE * (Y_BAR - LGD * pd_q)
            - LAMBDA * LEVERAGE * LGD * es - (LAMBDA / alpha) * p)


def dfl_context(pd_q):
    """Per-sample constants: annual PD, K_IRB, rho, and the composite derivative
    Kp = dK_IRB(pd_a(pd_q))/dpd_q by central finite difference (annualisation chain
    4(1-pd_q)^3 included automatically by differencing the composite map)."""
    pd_q = clip_pd_q(pd_q)
    pd_a = quarterly_pd_to_annual(pd_q)
    k = k_irb_from_annual_pd(pd_a)
    rho = dfl_rho(pd_a)
    h = np.maximum(1e-8, DFL_FD_REL * pd_q)
    k_up = k_irb_from_annual_pd(quarterly_pd_to_annual(pd_q + h))
    k_dn = k_irb_from_annual_pd(quarterly_pd_to_annual(pd_q - h))
    kp = (k_up - k_dn) / (2.0 * h)
    return pd_a, k, rho, kp


def solve_alpha_star(pd_q):
    """Vectorised forward + backward pass of the implicit allocation layer.
    Returns (alpha_star, dalpha_dpd). Boundary solutions get zero gradient.
    F is strictly decreasing in alpha wherever f_L(x) > 0 (strict concavity of U_hat),
    so: F(lo) <= 0 -> alpha* = 0; F(1) >= 0 -> alpha* = 1; else bisection on the
    unique interior root."""
    pd_q = clip_pd_q(np.atleast_1d(np.asarray(pd_q, dtype=float)))
    n = pd_q.shape[0]
    _, k, rho, kp = dfl_context(pd_q)

    lo = np.full(n, DFL_ALPHA_LO)
    hi = np.ones(n)
    f_lo = dfl_foc(lo, pd_q, k, rho)
    f_hi = dfl_foc(hi, pd_q, k, rho)
    at_one = f_hi >= 0.0
    at_zero = (~at_one) & (f_lo <= 0.0)
    interior = ~(at_one | at_zero)

    a_lo, a_hi = lo.copy(), hi.copy()
    for _ in range(DFL_BISECT_ITERS):
        mid = 0.5 * (a_lo + a_hi)
        f_mid = dfl_foc(mid, pd_q, k, rho)
        go_up = f_mid > 0.0
        a_lo = np.where(interior & go_up, mid, a_lo)
        a_hi = np.where(interior & ~go_up, mid, a_hi)
    alpha = np.where(at_one, 1.0, np.where(at_zero, 0.0, 0.5 * (a_lo + a_hi)))

    # IFT gradient at the interior optimum (zero at boundaries).
    a_safe = np.clip(alpha, DFL_ALPHA_LO, 1.0)
    x = dfl_threshold_x(a_safe, k)
    es, p, y_x, inside = vasicek_es_exceed(x, pd_q, rho)
    f_l = vasicek_pdf(x, pd_q, rho)
    a_probit = _ndtri(np.clip(pd_q, _PD_LO, 1.0 - _PD_LO))
    psi = np.where(inside, norm.cdf((y_x - np.sqrt(rho) * a_probit) / np.sqrt(1.0 - rho)),
                   np.where(x <= _X_EPS, 1.0, 0.0))
    dF_dalpha = -LAMBDA * f_l / (LEVERAGE * a_safe ** 3 * LGD)
    phi_a = np.maximum(norm.pdf(a_probit), 1e-300)
    dist_term = np.where(inside, norm.pdf(y_x) / (np.sqrt(rho) * phi_a), 0.0)
    dF_dpd = (-LEVERAGE * LGD
              - LAMBDA * LEVERAGE * LGD * (p * kp / LGD + psi)
              - (LAMBDA / a_safe) * (f_l * kp / LGD + dist_term))
    with np.errstate(divide="ignore", invalid="ignore"):
        grad = np.where(interior & (np.abs(dF_dalpha) > 1e-300), -dF_dpd / dF_dalpha, 0.0)
    grad = np.clip(np.nan_to_num(grad, nan=0.0, posinf=0.0, neginf=0.0),
                   -DFL_GRAD_CLIP, DFL_GRAD_CLIP)
    return alpha, grad


class ImplicitAllocation(torch.autograd.Function):
    """Differentiable allocation layer: forward solves the FOC by bisection; backward
    multiplies the incoming gradient by the closed-form IFT ratio dalpha*/dpd.
    The solver is never differentiated through (supervisor's grid warning): the gradient
    exists on paper, so a non-differentiable root-finder in the forward pass is fine."""

    @staticmethod
    def forward(ctx, pd_q_tensor):
        pd_np = pd_q_tensor.detach().cpu().numpy().astype(float).ravel()
        alpha, grad = solve_alpha_star(pd_np)
        ctx.save_for_backward(torch.as_tensor(grad, dtype=pd_q_tensor.dtype))
        return torch.as_tensor(alpha, dtype=pd_q_tensor.dtype).reshape(pd_q_tensor.shape)

    @staticmethod
    def backward(ctx, grad_output):
        (dalpha_dpd,) = ctx.saved_tensors
        return grad_output * dalpha_dpd.reshape(grad_output.shape)


def implicit_alpha(pd_q_tensor):
    return ImplicitAllocation.apply(pd_q_tensor)


def train_dfl_model(x_train, y_train_pd_q, c_next, k_next, u_oracle_next, model_type,
                    model_cfg, seed=None, return_history=False):
    """DFL training: identical mechanics to the black-box regret training loop (chronological val split,
    shuffled mini-batches, early stopping on validation regret) but the network output is
    a PD forecast passed through the implicit allocation layer:
        regret_t = U_oracle(t+1) - U(alpha*(pd_hat_t); c_{t+1}, K_{t+1}).
    Gradient chain (thesis): realised-utility subgradient (torch, eq. 3.43) x IFT ratio
    (custom backward, eq. IFT gradient) x network Jacobian (autodiff).
    Output bias initialised at logit(mean PD) of the pre-validation slice, exactly as the
    PTO forecaster (the DFL intermediate is a PD, not an allocation)."""
    seed = model_cfg.seed if seed is None else int(seed)
    torch.manual_seed(seed)
    np.random.seed(seed)
    n = x_train.shape[0]
    val_size = max(1, int(np.floor(model_cfg.val_frac * n)))
    train_size = n - val_size
    if train_size < 8:
        raise ValueError("Training set too small after validation split.")

    def to_t(arr):
        return torch.tensor(np.asarray(arr, dtype=float), dtype=torch.float32)

    x_tr, x_val = x_train[:train_size], x_train[train_size:]
    c_tr, c_val = c_next[:train_size], c_next[train_size:]
    k_tr, k_val = k_next[:train_size], k_next[train_size:]
    u_tr, u_val = u_oracle_next[:train_size], u_oracle_next[train_size:]

    train_ds = TensorDataset(to_t(x_tr), to_t(c_tr), to_t(k_tr), to_t(u_tr))
    loader_gen = torch.Generator().manual_seed(seed)
    train_loader = DataLoader(train_ds, batch_size=model_cfg.batch_size, shuffle=True, generator=loader_gen)

    model = PDForecaster(input_dim=x_train.shape[1], model_type=model_type, hidden_dim=model_cfg.hidden_dim)
    init_output_bias_for_target(model, y_train_pd_q[:train_size])
    optimiser = torch.optim.AdamW(model.parameters(), lr=model_cfg.lr, weight_decay=model_cfg.weight_decay)

    def regret_loss(score, cb, kb, ub):
        pd_q = torch.sigmoid(score).clamp(PD_Q_CLIP_LOW, PD_Q_CLIP_HIGH)
        alpha = implicit_alpha(pd_q)
        return torch.mean(ub - torch_allocation_utility(alpha, cb, kb))

    x_val_t, c_val_t, k_val_t, u_val_t = to_t(x_val), to_t(c_val), to_t(k_val), to_t(u_val)
    best_state, best_val, stale_epochs, history = None, float("inf"), 0, []

    for epoch in range(model_cfg.epochs):
        model.train()
        batch_losses = []
        for xb, cb, kb, ub in train_loader:
            optimiser.zero_grad()
            loss = regret_loss(model(xb), cb, kb, ub)
            loss.backward()
            optimiser.step()
            batch_losses.append(loss.item())

        model.eval()
        with torch.no_grad():
            score_val = model(x_val_t)
            pd_val = torch.sigmoid(score_val).clamp(PD_Q_CLIP_LOW, PD_Q_CLIP_HIGH).cpu().numpy()
        alpha_val, _ = solve_alpha_star(pd_val)
        val_regret = float(np.mean(u_val - allocation_utility_vec(alpha_val, c_val, k_val)))
        history.append({"epoch": epoch + 1, "model": f"dfl_{model_type}", "seed": seed,
                        "rho_eff": float("nan"),
                        "train_loss": float(np.mean(batch_losses)), "val_loss": val_regret,
                        "val_rmse_pd_q": float("nan"), "val_mae_pd_q": float("nan"),
                        "val_mean_pd_q_hat": float(np.mean(pd_val)),
                        "val_mean_alpha": float(np.mean(alpha_val))})

        if val_regret < best_val - 1e-12:
            best_val, stale_epochs = val_regret, 0
            best_state = {kk: v.detach().clone() for kk, v in model.state_dict().items()}
        else:
            stale_epochs += 1
        if stale_epochs >= model_cfg.patience:
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    return (model, pd.DataFrame(history)) if return_history else model


def allocation_utility_vec(alpha, c_q, k):
    """Vectorised realised utility over samples (alpha, c, k all 1-D)."""
    alpha = np.asarray(alpha, dtype=float)
    exposure = LEVERAGE * alpha * K1
    terminal_capital = K1 + exposure * (Y_BAR - np.asarray(c_q, dtype=float))
    available_buffer = terminal_capital - alpha * K1
    shortfall = np.maximum(np.asarray(k, dtype=float) * exposure - available_buffer, 0.0)
    return terminal_capital - LAMBDA * shortfall


def train_dfl_ensemble(x_train, y_train_pd_q, c_next, k_next, u_oracle_next, model_cfg,
                       seeds=DFL_SEEDS, return_history=False):
    models, first_hist = [], None
    for i, s in enumerate(seeds):
        want_hist = return_history and i == 0
        out = train_dfl_model(x_train, y_train_pd_q, c_next, k_next, u_oracle_next, "mlp",
                              model_cfg, seed=s, return_history=want_hist)
        if want_hist:
            model, first_hist = out
        else:
            model = out
        models.append(model)
    return models, first_hist


## Gradient audit

Every analytic ingredient of the implicit gradient is checked against an independent
numerical reference before any training runs. A wrong gradient here fails silently as
"DFL never beats PTO", indistinguishable from a genuine null result. The audit also
quantifies the one documented simplification: the gap between the IFT gradient (rho
fixed) and the full finite difference of the solver is exactly the omitted correlation
channel R'(pd), reported per test point for the thesis write-up.

In [ ]:
# ---------------------------------------------------------------------------------------
# Gradient audit (run once, before any training). A transcription error in the implicit
# gradient fails silently as "DFL never beats PTO", indistinguishable from a null result,
# so every analytic ingredient is checked against an independent reference here:
#   1. bvn_cdf vs scipy's bivariate normal CDF;
#   2. the FOC F vs a central finite difference of the smooth utility U_hat;
#   3. bisection alpha* vs a fine-grid argmax of U_hat (and, where available, the
#      Gauss-Hermite expected-utility grid rule used by PTO);
#   4. the IFT gradient vs a finite difference of the solver with rho FROZEN (exact
#      match expected), and vs the full finite difference (the gap IS the documented
#      omitted correlation channel R'(pd), reported for the thesis).
# ---------------------------------------------------------------------------------------
from scipy.stats import multivariate_normal as _mvn_audit

audit_pds = np.unique(np.round(np.quantile(
    df["pd_q_realised"].dropna().to_numpy(), [0.05, 0.25, 0.5, 0.75, 0.95]), 6))

print("1) bvn_cdf vs scipy:")
_bvn_err = 0.0
for _a, _b, _r in [(-2.1, 0.5, 0.4), (0.0, 0.0, 0.49), (-1.0, -1.0, 0.35), (1.5, -0.7, 0.45)]:
    _ref = float(_mvn_audit(mean=[0, 0], cov=[[1, _r], [_r, 1]]).cdf([_a, _b]))
    _bvn_err = max(_bvn_err, abs(float(bvn_cdf(_a, _b, _r)[0]) - _ref))
print(f"   max abs err = {_bvn_err:.2e}")
assert _bvn_err < 1e-8

print("2) FOC F vs central FD of U_hat:")
_foc_err = 0.0
for _pd in audit_pds:
    _, _k, _rho, _ = dfl_context(np.array([_pd]))
    for _al in [0.05, 0.2, 0.5, 0.9]:
        _h = 1e-6
        _fd = (dfl_smooth_utility(np.array([_al + _h]), np.array([_pd]), _k, _rho)[0]
               - dfl_smooth_utility(np.array([_al - _h]), np.array([_pd]), _k, _rho)[0]) / (2 * _h)
        _an = float(dfl_foc(np.array([_al]), np.array([_pd]), _k, _rho)[0])
        _foc_err = max(_foc_err, abs(_fd - _an) / max(1.0, abs(_an)))
print(f"   max rel err = {_foc_err:.2e}")
assert _foc_err < 1e-4

print("3) bisection alpha* vs fine-grid argmax of U_hat:")
_agrid = np.linspace(DFL_ALPHA_LO, 1.0, 200001)
for _pd in audit_pds:
    _, _k, _rho, _ = dfl_context(np.array([_pd]))
    _ug = dfl_smooth_utility(_agrid, np.full_like(_agrid, _pd),
                             np.full_like(_agrid, _k[0]), np.full_like(_agrid, _rho[0]))
    _ag = float(_agrid[np.argmax(_ug)])
    _ab, _g = solve_alpha_star(np.array([_pd]))
    _line = f"   pd={_pd:.5f}  grid={_ag:.5f}  bisect={float(_ab[0]):.5f}  dalpha/dpd={float(_g[0]):+9.3f}"
    try:
        _aeu, _ = optimise_alpha_expected(float(_pd))
        _line += f"  (PTO EU grid rule: {_aeu:.2f})"
    except NameError:
        pass
    print(_line)
    assert abs(_ag - float(_ab[0])) < 2e-4 or (_ag >= 1.0 - 1e-4 and float(_ab[0]) == 1.0) \
        or (_ag <= DFL_ALPHA_LO * 2 and float(_ab[0]) == 0.0)

print("4) IFT gradient vs FD of the solver (rho frozen -> exact; full FD gap = omitted R' channel):")
_orig_dfl_rho = dfl_rho
for _pd in audit_pds:
    _a0, _g0 = solve_alpha_star(np.array([_pd]))
    if not (0.0 < float(_a0[0]) < 1.0):
        print(f"   pd={_pd:.5f}  alpha*={float(_a0[0]):.2f} (boundary; gradient correctly zero: {float(_g0[0]):.1f})")
        continue
    _h = 1e-7
    _rho0 = float(_orig_dfl_rho(quarterly_pd_to_annual(np.array([_pd])))[0])
    dfl_rho = lambda pd_a, _r=_rho0: np.full_like(np.atleast_1d(np.asarray(pd_a, dtype=float)), _r)
    _fp, _ = solve_alpha_star(np.array([_pd + _h])); _fm, _ = solve_alpha_star(np.array([_pd - _h]))
    _fd_frozen = float((_fp[0] - _fm[0]) / (2 * _h))
    dfl_rho = _orig_dfl_rho
    _fp2, _ = solve_alpha_star(np.array([_pd + _h])); _fm2, _ = solve_alpha_star(np.array([_pd - _h]))
    _fd_full = float((_fp2[0] - _fm2[0]) / (2 * _h))
    _ift = float(_g0[0])
    _rel = abs(_fd_frozen - _ift) / max(1e-12, abs(_ift))
    print(f"   pd={_pd:.5f}  IFT={_ift:+9.3f}  FD(rho frozen)={_fd_frozen:+9.3f} (rel {_rel:.1e})"
          f"  FD(full)={_fd_full:+9.3f}  omitted-channel share={abs(_fd_full - _ift) / max(1e-12, abs(_fd_full)):.1%}")
    assert _rel < 1e-3, (_pd, _fd_frozen, _ift)
dfl_rho = _orig_dfl_rho
print("Gradient audit passed: forward solver and closed-form IFT backward are consistent.")


## Expanding-window evaluation

At each test date the scaler and models are refit on observations strictly before the
feature date. Three PD predictors run through the identical decision pipeline: the linear
model, the MLP seed ensemble (mean prediction), and a persistence baseline
`pd_q_hat = c_q_t / LGD`. Persistence is the reference point: a trained model that does not
beat it on regret adds no decision value over "assume next quarter looks like this quarter".

Two splits run as separate experiments: `modern` (last 25% of the sample) and `stress`
(test targets from 2006Q1, so the GFC is out-of-sample). The modern test window is contained
in the stress one; they are not independent samples.

The DFL models are trained at each test date on the **same expanding window and the same
fitted scaler** as the PTO models (`dfl_linear` single seed, `dfl_mlp` seed ensemble whose
PD forecasts are averaged before the decision - the PTO convention, preserving the
auditable intermediate). DFL enters result collection through `build_result_row_dfl`: PD
fields populated exactly as for PTO, the decision from the implicit layer (continuous
optimum kept in `alpha_hat_raw_dfl`, grid-snapped `alpha_hat` for evaluation). Five
predictors therefore run through the identical decision pipeline. Runtime note: DFL adds
two regret-trained model fits per window on top of the PTO fits, so expect roughly the
black-box notebook's runtime.

In [ ]:
def build_result_row(base_row, pd_q_hat, model_name, split_name, pd_q_hat_std=float("nan")):
    pd_q_hat = float(clip_pd_q(pd_q_hat))
    c_q_hat = float(LGD * pd_q_hat)
    pd_a_hat = float(quarterly_pd_to_annual(pd_q_hat))
    c_q_realised = float(base_row["c_q_realised"])
    pd_q_realised = float(base_row["pd_q_realised"])
    pd_a_realised = float(base_row["pd_a_realised"])

    alpha_hat, _ = optimise_alpha_grid(c_q=c_q_hat, pd_a=pd_a_hat)
    alpha_oracle, utility_oracle = optimise_alpha_grid(c_q=c_q_realised, pd_a=pd_a_realised)
    realised_components = decision_components(alpha_hat, c_q=c_q_realised, pd_a=pd_a_realised)
    oracle_components = decision_components(alpha_oracle, c_q=c_q_realised, pd_a=pd_a_realised)
    predicted_components = decision_components(alpha_hat, c_q=c_q_hat, pd_a=pd_a_hat)
    regret = utility_oracle - realised_components["utility"]

    return {"split": split_name, "model": model_name, "date": pd.to_datetime(base_row["target_date"]),
            "feature_date": pd.to_datetime(base_row[DATE_COL]), "c_annual_raw": float(base_row["c_annual_raw_next"]),
            "c_q_lag1": float(base_row["c_q_lag1"]),
            "c_q_realised": c_q_realised, "pd_q_realised": pd_q_realised, "pd_a_realised": pd_a_realised,
            "pd_q_hat": pd_q_hat, "pd_q_hat_std": float(pd_q_hat_std), "pd_a_hat": pd_a_hat, "c_q_hat": c_q_hat,
            "k_irb_realised": float(k_irb_from_annual_pd(pd_a_realised)), "k_irb_hat": float(k_irb_from_annual_pd(pd_a_hat)),
            "alpha_oracle": alpha_oracle, "alpha_hat": alpha_hat, "utility_oracle": utility_oracle,
            "utility": realised_components["utility"], "regret": regret,
            "shortfall_oracle": oracle_components["shortfall"],
            "shortfall_realised_at_hat": realised_components["shortfall"],
            "shortfall_predicted_at_hat": predicted_components["shortfall"]}


def build_result_row_dfl(base_row, pd_q_hat, model_name, split_name, pd_q_hat_std=float("nan")):
    """DFL counterpart of build_result_row: enters the harness at the PD level exactly
    like PTO (the DFL model keeps an auditable PD intermediate), but the allocation is
    the implicit-layer optimum alpha*(pd_q_hat) of the smooth forecast-based objective
    (bisection on the FOC), snapped to the shared grid for evaluation only. The raw
    continuous optimum is kept in `alpha_hat_raw_dfl` for diagnostics."""
    pd_q_hat = float(clip_pd_q(pd_q_hat))
    c_q_hat = float(LGD * pd_q_hat)
    pd_a_hat = float(quarterly_pd_to_annual(pd_q_hat))
    c_q_realised = float(base_row["c_q_realised"])
    pd_q_realised = float(base_row["pd_q_realised"])
    pd_a_realised = float(base_row["pd_a_realised"])

    _alpha_raw, _ = solve_alpha_star(np.array([pd_q_hat]))
    alpha_hat_raw = float(_alpha_raw[0])
    alpha_hat = float(snap_alpha(alpha_hat_raw)[0])
    alpha_oracle, utility_oracle = optimise_alpha_grid(c_q=c_q_realised, pd_a=pd_a_realised)
    realised_components = decision_components(alpha_hat, c_q=c_q_realised, pd_a=pd_a_realised)
    oracle_components = decision_components(alpha_oracle, c_q=c_q_realised, pd_a=pd_a_realised)
    predicted_components = decision_components(alpha_hat, c_q=c_q_hat, pd_a=pd_a_hat)
    regret = utility_oracle - realised_components["utility"]

    return {"split": split_name, "model": model_name, "date": pd.to_datetime(base_row["target_date"]),
            "feature_date": pd.to_datetime(base_row[DATE_COL]), "c_annual_raw": float(base_row["c_annual_raw_next"]),
            "c_q_lag1": float(base_row["c_q_lag1"]),
            "c_q_realised": c_q_realised, "pd_q_realised": pd_q_realised, "pd_a_realised": pd_a_realised,
            "pd_q_hat": pd_q_hat, "pd_q_hat_std": float(pd_q_hat_std), "pd_a_hat": pd_a_hat, "c_q_hat": c_q_hat,
            "k_irb_realised": float(k_irb_from_annual_pd(pd_a_realised)), "k_irb_hat": float(k_irb_from_annual_pd(pd_a_hat)),
            "alpha_oracle": alpha_oracle, "alpha_hat": alpha_hat,
            "alpha_hat_raw_dfl": alpha_hat_raw,
            "utility_oracle": utility_oracle,
            "utility": realised_components["utility"], "regret": regret,
            "shortfall_oracle": oracle_components["shortfall"],
            "shortfall_realised_at_hat": realised_components["shortfall"],
            "shortfall_predicted_at_hat": predicted_components["shortfall"]}


def resolve_first_test_idx(df, split_cfg):
    if "initial_train_frac" in split_cfg:
        first_test_idx = int(np.floor(len(df) * float(split_cfg["initial_train_frac"])))
    elif "first_test_date" in split_cfg:
        mask = (df["target_date"] >= pd.Timestamp(split_cfg["first_test_date"])).to_numpy()
        if not mask.any():
            raise ValueError("first_test_date is after the last available target date.")
        first_test_idx = int(np.flatnonzero(mask)[0])
    else:
        raise ValueError("Split config needs 'initial_train_frac' or 'first_test_date'.")
    if first_test_idx < 30 or len(df) - first_test_idx < 5:
        raise ValueError(f"Split leaves too little data: first_test_idx={first_test_idx} of {len(df)}.")
    return first_test_idx


def run_expanding_window_pd_irb(df, feature_cols, split_name, split_cfg, model_cfg=MODEL_CFG, return_history=True):
    first_test_idx = resolve_first_test_idx(df, split_cfg)
    total = len(df) - first_test_idx
    print(f"[{split_name}] first test target: {pd.Timestamp(df['target_date'].iloc[first_test_idx]).date()}, "
          f"{total} windows, {first_test_idx} initial training obs")
    records, histories = [], []
    t0 = time.time()
    for test_idx in range(first_test_idx, len(df)):
        train_df = df.iloc[:test_idx]
        test_row = df.iloc[test_idx]
        scaler = StandardScaler()
        x_train = scaler.fit_transform(train_df[feature_cols].to_numpy(dtype=float))
        y_train = train_df["target_pd_q_next"].to_numpy(dtype=float)
        x_test = scaler.transform(test_row[feature_cols].to_numpy(dtype=float).reshape(1, -1))

        linear_model, linear_hist = train_pd_model(x_train, y_train, "linear", model_cfg, return_history=True)
        records.append(build_result_row(test_row, float(predict_pd_q(linear_model, x_test)[0]), "linear", split_name))

        mlp_models, mlp_hist = train_mlp_ensemble(x_train, y_train, model_cfg, return_history=True)
        mlp_mean, mlp_std = predict_pd_q_ensemble(mlp_models, x_test)
        records.append(build_result_row(test_row, float(mlp_mean[0]), "mlp", split_name, pd_q_hat_std=float(mlp_std[0])))

        # Persistence baseline: current-quarter implied PD carried forward, no training.
        records.append(build_result_row(test_row, float(clip_pd_q(test_row["c_q"] / LGD)), "persistence", split_name))

        # DFL: same window and scaler; PD forecaster trained on realised regret through
        # the implicit allocation layer. Training signal (hindsight outcomes, identical
        # to the black-box benchmark): realised c_{t+1}, k_irb(pd_a_realised_{t+1}),
        # foresight-oracle utility. y_train is used only for output-bias initialisation.
        c_next_tr = train_df["c_q_realised"].to_numpy(dtype=float)
        k_next_tr = train_df["k_irb_realised_next"].to_numpy(dtype=float)
        u_oracle_tr = train_df["utility_oracle_next"].to_numpy(dtype=float)
        dfl_linear_model, dfl_linear_hist = train_dfl_model(
            x_train, y_train, c_next_tr, k_next_tr, u_oracle_tr, "linear",
            model_cfg, return_history=True)
        records.append(build_result_row_dfl(test_row, float(predict_pd_q(dfl_linear_model, x_test)[0]),
                                            "dfl_linear", split_name))
        dfl_mlp_models, dfl_mlp_hist = train_dfl_ensemble(
            x_train, y_train, c_next_tr, k_next_tr, u_oracle_tr, model_cfg,
            return_history=True)
        dfl_mean, dfl_std = predict_pd_q_ensemble(dfl_mlp_models, x_test)
        records.append(build_result_row_dfl(test_row, float(dfl_mean[0]), "dfl_mlp", split_name,
                                            pd_q_hat_std=float(dfl_std[0])))

        if return_history:
            for hist in (linear_hist, mlp_hist, dfl_linear_hist, dfl_mlp_hist):
                if hist is not None:
                    hist = hist.copy()
                    hist["split"] = split_name
                    hist["test_date"] = pd.to_datetime(test_row["target_date"])
                    hist["train_end_date"] = pd.to_datetime(train_df[DATE_COL].iloc[-1])
                    histories.append(hist)

        done = test_idx - first_test_idx + 1
        if done % 5 == 0 or test_idx == len(df) - 1:
            print(f"[{split_name}] completed {done}/{total} windows ({time.time() - t0:.0f}s elapsed)")
    pred_long = pd.DataFrame(records).sort_values(["date", "model"]).reset_index(drop=True)
    history = pd.concat(histories, ignore_index=True) if histories else pd.DataFrame()
    return pred_long, history


split_frames, split_histories = [], []
for split_name, split_cfg in EVAL_SPLITS.items():
    pl, th = run_expanding_window_pd_irb(df, feature_cols, split_name, split_cfg)
    split_frames.append(pl)
    split_histories.append(th)
pred_long = pd.concat(split_frames, ignore_index=True)
training_history = pd.concat([h for h in split_histories if not h.empty], ignore_index=True)
display(pred_long.head())

## Result collection

One wide, thesis-facing frame per split; `_persistence` columns sit alongside the model
columns so the baseline flows through every downstream table and figure.

In [ ]:
base_cols = ["date", "c_annual_raw", "c_q_lag1", "c_q_realised", "pd_q_realised", "pd_a_realised",
             "k_irb_realised", "alpha_oracle", "utility_oracle"]
metric_cols = ["pd_q_hat", "pd_a_hat", "c_q_hat", "k_irb_hat", "alpha_hat", "utility", "regret"]
MODEL_ORDER = ["linear", "mlp", "dfl_linear", "dfl_mlp", "persistence"]
PD_MODELS = ["linear", "mlp", "dfl_linear", "dfl_mlp", "persistence"]                       # models with a PD intermediate
DFL_MODELS = ["dfl_linear", "dfl_mlp"]                    # alpha-level models (PD fields NaN)

def make_wide_results(pred_long, split_name):
    sub = pred_long[pred_long["split"] == split_name]
    wide = sub.pivot(index="date", columns="model", values=metric_cols).sort_index()
    wide.columns = [f"{metric}_{model}" for metric, model in wide.columns]
    wide = wide.reset_index()
    base = sub[sub["model"] == "linear"][base_cols].copy()
    out = base.merge(wide, on="date", how="left")
    ordered = base_cols + [f"{m}_{mod}" for m in metric_cols for mod in MODEL_ORDER]
    return out[ordered].sort_values("date").reset_index(drop=True)

results_by_split = {}
for split_name in EVAL_SPLITS:
    res = make_wide_results(pred_long, split_name)
    results_by_split[split_name] = res
    output_csv = RESULTS_DIR / f"dfl_quarterly_pd_irb_predictions_{split_name}.csv"
    res.to_csv(output_csv, index=False)
    print(f"[{split_name}] saved {len(res)} rows to {output_csv.relative_to(ROOT)}")
display(results_by_split["modern"].head())

## Out-of-sample accuracy and model comparison

RMSE/MAE/bias on quarterly PD, decision metrics, and pairwise Diebold-Mariano tests with a
Newey-West long-run variance and the Harvey et al. small-sample adjustment. A negative DM
statistic means the first model of the pair has the lower loss. The `regret` columns apply
the same HAC mean-difference machinery to per-period regret — not a classic forecast-loss
DM, so read it as a mean-difference test. With ~36 observations in the modern split, treat
p-values as indicative rather than decisive; the same tests can later compare this PTO model
against the DFL and decision-focused variants on their shared test dates.
For the DFL model the PD-accuracy columns and the DM test on squared PD errors are NaN
**by construction** - there is no PD to score. The regret columns, the shortfall bind
share and the DM test on regret are the comparison that matters; the added pairs test the
DFL model against its PTO counterpart of equal capacity and against persistence, on
shared test dates.


In [ ]:
def newey_west_lrv(d, lags):
    d = np.asarray(d, dtype=float)
    d = d - d.mean()
    n = len(d)
    v = float(np.mean(d * d))
    for k in range(1, min(lags, n - 1) + 1):
        w = 1.0 - k / (lags + 1.0)
        v += 2.0 * w * float(np.mean(d[k:] * d[:-k]))
    return v / n


def dm_test(loss_a, loss_b, nw_lags=4):
    # Diebold-Mariano mean-difference test; negative stat => first series has lower loss.
    d = np.asarray(loss_a, dtype=float) - np.asarray(loss_b, dtype=float)
    n = len(d)
    lrv = newey_west_lrv(d, nw_lags)
    if not np.isfinite(lrv) or lrv <= 0:
        return float("nan"), float("nan")
    stat = (d.mean() / np.sqrt(lrv)) * np.sqrt((n - 1.0) / n)  # Harvey adjustment, h=1
    p_value = 2.0 * stats.t.sf(abs(stat), df=n - 1)
    return float(stat), float(p_value)


def forecast_metric_table(sub):
    rows = []
    for model_name, g in sub.groupby("model"):
        err = g["pd_q_hat"].to_numpy() - g["pd_q_realised"].to_numpy()
        rows.append({"model": model_name, "n": int(len(g)),
                     "rmse_pd_q": float(np.sqrt(np.mean(err ** 2))),
                     "mae_pd_q": float(np.mean(np.abs(err))),
                     "mean_bias_pd_q": float(np.mean(err)),
                     "mean_regret": float(g["regret"].mean()),
                     "median_regret": float(g["regret"].median()),
                     "mean_utility": float(g["utility"].mean()),
                     "shortfall_bind_share": float((g["shortfall_realised_at_hat"] > 1e-12).mean())})
    return pd.DataFrame(rows).set_index("model").loc[MODEL_ORDER]

DM_PAIRS = [("linear", "persistence"), ("mlp", "persistence"), ("linear", "mlp"),
            ("dfl_linear", "linear"), ("dfl_mlp", "mlp"),
            ("dfl_linear", "persistence"), ("dfl_mlp", "persistence"),
            ("dfl_linear", "dfl_mlp")]

for split_name in EVAL_SPLITS:
    sub = pred_long[pred_long["split"] == split_name]
    print(f"=== {split_name} split ===")
    metrics = forecast_metric_table(sub)
    display(metrics)
    metrics.to_csv(RESULTS_DIR / f"dfl_quarterly_pd_irb_metrics_{split_name}.csv")

    by_model = {m: g.sort_values("date") for m, g in sub.groupby("model")}
    dm_rows = []
    for a, b in DM_PAIRS:
        # DFL keeps a PD intermediate, so the PD-accuracy DM applies to every pair.
        if False:
            dm_se, p_se = float("nan"), float("nan")
        else:
            err2_a = (by_model[a]["pd_q_hat"].to_numpy() - by_model[a]["pd_q_realised"].to_numpy()) ** 2
            err2_b = (by_model[b]["pd_q_hat"].to_numpy() - by_model[b]["pd_q_realised"].to_numpy()) ** 2
            dm_se, p_se = dm_test(err2_a, err2_b)
        dm_rg, p_rg = dm_test(by_model[a]["regret"].to_numpy(), by_model[b]["regret"].to_numpy())
        dm_rows.append({"pair": f"{a} vs {b}", "dm_sq_error": dm_se, "p_sq_error": p_se,
                        "dm_regret": dm_rg, "p_regret": p_rg})
    dm_table = pd.DataFrame(dm_rows).set_index("pair")
    display(dm_table)
    dm_table.to_csv(RESULTS_DIR / f"dfl_quarterly_pd_irb_dm_tests_{split_name}.csv")

## Diagnostics and sanity checks
PD-side checks apply to PD models only. DFL guards: PD-intermediate fields must be
NaN, alphas must sit on the shared decision grid, and regret must be >= 0 for every model.


In [ ]:
def describe_columns(frame, cols):
    return frame[cols].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).T

print(f"Charge-off conversion: {CHARGE_OFF_CONVERSION!r}; percent input: {CHARGE_OFF_IS_PERCENT}; "
      f"target transform: {TARGET_TRANSFORM!r}; PD floor applied: {APPLY_PD_FLOOR} ({PD_FLOOR:.2%}); "
      f"MLP seeds: {MLP_SEEDS}; DFL seeds: {DFL_SEEDS}")
print(f"PTO loss: {LOSS!r}; NLL rho mode: {NLL_RHO_MODE}; NLL rho scale (basel mode): {NLL_RHO_SCALE}")
print("DFL: PD forecaster trained on realised regret through the implicit layer; "
      "decision by bisection on the smooth objective; grid snap at evaluation only.")
if LOSS == "nll_vasicek" and NLL_RHO_MODE == "auto" and not training_history.empty and "rho_eff" in training_history.columns:
    print("Estimated rho_eff across windows (PD models only):")
    display(training_history[training_history["model"].isin(["linear", "mlp"])]
            .groupby("model")["rho_eff"].describe()[["mean", "min", "max"]])

pd_rows = pred_long[pred_long["model"].isin(PD_MODELS)]
dfl_rows = pred_long[pred_long["model"].isin(DFL_MODELS)]
assert len(dfl_rows) and len(pd_rows)

# --- Shared realised-outcome checks (all rows) ---
assert np.allclose(pred_long["pd_q_realised"], clip_pd_q(pred_long["c_q_realised"] / LGD))
assert np.allclose(pred_long["pd_a_realised"], quarterly_pd_to_annual(pred_long["pd_q_realised"]), atol=1e-12)
for col in ["pd_q_realised", "pd_a_realised"]:
    vals = pred_long[col].to_numpy()
    assert np.isfinite(vals).all() and (vals > 0.0).all() and (vals < 1.0).all(), col
vals = pred_long["k_irb_realised"].to_numpy()
assert np.isfinite(vals).all() and (vals >= 0.0).all()

# --- PD-side checks (PD models only; the DFL model has no PD intermediate) ---
for col in ["pd_q_hat", "pd_a_hat"]:
    vals = pd_rows[col].to_numpy()
    assert np.isfinite(vals).all() and (vals > 0.0).all() and (vals < 1.0).all(), col
vals = pd_rows["k_irb_hat"].to_numpy()
assert np.isfinite(vals).all() and (vals >= 0.0).all()
assert np.allclose(pd_rows["pd_a_hat"], quarterly_pd_to_annual(pd_rows["pd_q_hat"]), atol=1e-12)
assert np.allclose(pd_rows["k_irb_hat"], k_irb_from_annual_pd(pd_rows["pd_a_hat"].to_numpy()), atol=1e-12)
print("IRB check passed: k_irb_from_annual_pd is called on annualised PD columns only.")

# --- DFL guards: PD-level entry with an implicit-layer decision ---
for col in ["pd_q_hat", "pd_a_hat", "c_q_hat", "k_irb_hat"]:
    vals = dfl_rows[col].to_numpy()
    assert np.isfinite(vals).all(), f"{col} must be finite for DFL rows (PD intermediate)"
dfl_alpha = dfl_rows["alpha_hat"].to_numpy()
assert np.isfinite(dfl_alpha).all() and (dfl_alpha >= 0.0).all() and (dfl_alpha <= 1.0).all()
assert np.isclose(np.abs(ALPHA_GRID[None, :] - dfl_alpha[:, None]).min(axis=1), 0.0, atol=1e-12).all(), \
    "DFL alphas must be snapped to the shared decision grid."
dfl_alpha_raw = dfl_rows["alpha_hat_raw_dfl"].to_numpy()
assert np.isfinite(dfl_alpha_raw).all() and (np.abs(dfl_alpha - dfl_alpha_raw) <= 0.005 + 1e-12).all(), \
    "Grid snap moved an alpha by more than half a grid step."
# Reproducibility: the stored continuous optimum must re-solve identically.
_resolved, _ = solve_alpha_star(dfl_rows["pd_q_hat"].to_numpy())
assert np.allclose(_resolved, dfl_alpha_raw, atol=1e-8), "solve_alpha_star is not reproducible."
print("DFL guards passed (finite PD fields, alphas on grid, solver reproducible).")

min_regret = pred_long["regret"].min()
if min_regret < -1e-10:
    raise AssertionError(f"Meaningfully negative regret found: {min_regret}")
print(f"Minimum regret across splits and models (DFL model included): {min_regret:.3e}")

summary_cols = (["c_annual_raw", "c_q_lag1", "c_q_realised", "pd_q_realised"]
                + [f"pd_q_hat_{m}" for m in PD_MODELS]
                + ["pd_a_realised"] + [f"pd_a_hat_{m}" for m in PD_MODELS]
                + ["k_irb_realised"] + [f"k_irb_hat_{m}" for m in PD_MODELS]
                + ["alpha_oracle"] + [f"alpha_hat_{m}" for m in MODEL_ORDER]
                + [f"regret_{m}" for m in MODEL_ORDER])
for split_name, res in results_by_split.items():
    print(f"=== {split_name} split summary ===")
    display(describe_columns(res, summary_cols))

shortfall_summary = pred_long.groupby(["split", "model"])[["shortfall_realised_at_hat", "shortfall_predicted_at_hat"]].agg(
    ["mean", "max", lambda s: float((s > 1e-12).mean())])
shortfall_summary = shortfall_summary.rename(columns={"<lambda_0>": "bind_share"})
print(f"Decision constants: LEVERAGE={LEVERAGE}, Y_BAR={Y_BAR}, LAMBDA={LAMBDA}")
display(shortfall_summary)

mlp_std = pred_long[pred_long["model"] == "mlp"].groupby("split")["pd_q_hat_std"].describe()
print("Cross-seed std of PTO MLP ensemble PD predictions (per split):")
display(mlp_std)
dfl_std_desc = pred_long[pred_long["model"] == "dfl_mlp"].groupby("split")["pd_q_hat_std"].describe()
print("Cross-seed std of DFL MLP ensemble PD predictions (per split):")
display(dfl_std_desc)


## Plots

The full figure set is produced per split into `figures/dfl_pd_irb_v2/<split>/`.
`PLOT_MODELS` selects which models appear; PD-based panels show PD models only and are
skipped entirely if no PD model is selected. The allocation and regret panels include the
DFL model, which also gets an alpha-vs-oracle scatter.


In [ ]:
plt.rcdefaults()
plt.rcParams.update({"font.size": 10, "axes.linewidth": 0.8, "xtick.direction": "out", "ytick.direction": "out",
                     "xtick.major.size": 3, "ytick.major.size": 3, "font.sans-serif": ["DejaVu Sans"]})
MODEL_COLORS = {"linear": "#2166ac", "mlp": "#d6604d", "dfl_linear": "#7b3294",
                "dfl_mlp": "#e7298a", "persistence": "#888888"}
MODEL_LW = {"linear": 1.4, "mlp": 1.4, "dfl_linear": 1.4, "dfl_mlp": 1.4, "persistence": 1.0}
MODEL_LABELS = {"linear": "PTO linear", "mlp": "PTO MLP", "dfl_linear": "DFL linear",
                "dfl_mlp": "DFL MLP", "persistence": "Persistence"}
ORACLE_COLOR = "#1a1a1a"
REAL_COLOR = "#333333"

# Which models appear in the static figures. Any subset of MODEL_ORDER works, including
# DFL-only: PD-based panels are skipped when no PD model is selected.
PLOT_MODELS = ["linear", "mlp", "dfl_linear", "dfl_mlp", "persistence"]
PLOT_MODELS_PD = [m for m in PLOT_MODELS if m in PD_MODELS]
PLOT_MODELS_DFL = [m for m in PLOT_MODELS if m in DFL_MODELS]

def format_date_axis(ax, year_step=2):
    ax.xaxis.set_major_locator(mdates.YearLocator(year_step))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.grid(True, alpha=0.28)


def _model_lines(ax, plot_df, col_prefix, suffix="", models=None):
    for m in (PLOT_MODELS if models is None else models):
        ax.plot(plot_df["date"], plot_df[f"{col_prefix}_{m}"], color=MODEL_COLORS[m],
                lw=MODEL_LW[m], label=f"{MODEL_LABELS[m]}{suffix}")


def plot_split(res, split_name):
    fig_dir = FIGURES_DIR / split_name
    fig_dir.mkdir(parents=True, exist_ok=True)
    plot_df = res.sort_values("date").copy()
    year_step = 2 if len(plot_df) <= 60 else 4

    if PLOT_MODELS_PD:
        # 1. Quarterly PD predictions (PD models only)
        fig, ax = plt.subplots(figsize=(10, 4))
        ax.plot(plot_df["date"], plot_df["pd_q_realised"], color=REAL_COLOR, lw=2, ls="--", label="Realised quarterly PD")
        _model_lines(ax, plot_df, "pd_q_hat", " predicted", models=PLOT_MODELS_PD)
        ax.set_ylabel("Quarterly PD"); ax.set_title(f"Quarterly PD predictions ({split_name}; PD models only)")
        ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
        format_date_axis(ax, year_step); ax.legend(frameon=False); plt.tight_layout()
        plt.savefig(fig_dir / "quarterly_pd_predictions.png", dpi=200, bbox_inches="tight"); plt.show()

        # 2. Quarterly charge-off predictions (PD models only)
        fig, ax = plt.subplots(figsize=(10, 4))
        ax.plot(plot_df["date"], plot_df["c_q_realised"], color=REAL_COLOR, lw=2, ls="--", label="Realised quarterly charge-off")
        _model_lines(ax, plot_df, "c_q_hat", " implied", models=PLOT_MODELS_PD)
        ax.set_ylabel("Quarterly charge-off"); ax.set_title(f"Realised and implied quarterly charge-offs ({split_name}; PD models only)")
        ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
        format_date_axis(ax, year_step); ax.legend(frameon=False); plt.tight_layout()
        plt.savefig(fig_dir / "quarterly_chargeoff_predictions.png", dpi=200, bbox_inches="tight"); plt.show()

        # 3. Annualised PD for IRB (PD models only)
        fig, ax = plt.subplots(figsize=(10, 4))
        ax.plot(plot_df["date"], plot_df["pd_a_realised"], color=REAL_COLOR, lw=2, ls="--", label="Realised annualised PD")
        _model_lines(ax, plot_df, "pd_a_hat", " annualised", models=PLOT_MODELS_PD)
        ax.set_ylabel("Annualised PD"); ax.set_title(f"Annualised PD used for Basel IRB ({split_name}; PD models only)")
        ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
        format_date_axis(ax, year_step); ax.legend(frameon=False); plt.tight_layout()
        plt.savefig(fig_dir / "annualised_pd_irb.png", dpi=200, bbox_inches="tight"); plt.show()

        # 4. IRB capital requirement (PD models only)
        fig, ax = plt.subplots(figsize=(10, 4))
        ax.plot(plot_df["date"], plot_df["k_irb_realised"], color=REAL_COLOR, lw=2, ls="--", label="Realised K_IRB")
        _model_lines(ax, plot_df, "k_irb_hat", " K_IRB", models=PLOT_MODELS_PD)
        ax.set_ylabel("K_IRB"); ax.set_title(f"Basel IRB capital requirement ({split_name}; PD models only)")
        ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
        format_date_axis(ax, year_step); ax.legend(frameon=False); plt.tight_layout()
        plt.savefig(fig_dir / "irb_capital_requirement.png", dpi=200, bbox_inches="tight"); plt.show()
    else:
        print(f"[{split_name}] no PD models in PLOT_MODELS - skipping PD, charge-off, K_IRB and PD-scatter figures.")

    # 5. Allocation comparison (all selected models)
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(plot_df["date"], plot_df["alpha_oracle"], color=ORACLE_COLOR, lw=2, ls="--", label="Oracle")
    _model_lines(ax, plot_df, "alpha_hat")
    ax.set_ylabel(r"Allocation $\alpha$"); ax.set_title(f"Allocations versus oracle ({split_name})")
    format_date_axis(ax, year_step); ax.legend(frameon=False, fontsize=8, ncol=2); plt.tight_layout()
    plt.savefig(fig_dir / "allocation_comparison.png", dpi=200, bbox_inches="tight"); plt.show()

    # 6. Regret and cumulative regret (all selected models)
    fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
    for m in PLOT_MODELS:
        axes[0].plot(plot_df["date"], plot_df[f"regret_{m}"], color=MODEL_COLORS[m], lw=MODEL_LW[m], label=MODEL_LABELS[m])
        axes[1].plot(plot_df["date"], plot_df[f"regret_{m}"].cumsum(), color=MODEL_COLORS[m], lw=MODEL_LW[m], label=MODEL_LABELS[m])
    axes[0].set_ylabel("Regret"); axes[0].legend(frameon=False, fontsize=8)
    axes[1].set_ylabel("Cumulative regret"); axes[1].set_xlabel("Date"); axes[1].legend(frameon=False, fontsize=8)
    for ax in axes:
        format_date_axis(ax, year_step)
    fig.suptitle(f"Regret ({split_name})"); plt.tight_layout()
    plt.savefig(fig_dir / "regret.png", dpi=200, bbox_inches="tight"); plt.show()

    # 7. Compact 4-panel thesis figure (PD rows show PD models only; empty when none selected)
    fig, axes = plt.subplots(4, 1, figsize=(10, 11), sharex=True)
    fig.subplots_adjust(hspace=0.08)
    axes[0].plot(plot_df["date"], plot_df["pd_q_realised"], color=REAL_COLOR, lw=2, ls="--", label="Realised")
    _model_lines(axes[0], plot_df, "pd_q_hat", models=PLOT_MODELS_PD)
    axes[0].set_ylabel("Quarterly PD"); axes[0].yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
    axes[1].plot(plot_df["date"], plot_df["c_q_realised"], color=REAL_COLOR, lw=2, ls="--", label="Realised")
    _model_lines(axes[1], plot_df, "c_q_hat", models=PLOT_MODELS_PD)
    axes[1].set_ylabel("Quarterly charge-off"); axes[1].yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
    axes[2].plot(plot_df["date"], plot_df["alpha_oracle"], color=ORACLE_COLOR, lw=2, ls="--", label="Oracle")
    _model_lines(axes[2], plot_df, "alpha_hat")
    axes[2].set_ylabel(r"$\alpha$")
    for m in PLOT_MODELS:
        axes[3].plot(plot_df["date"], plot_df[f"regret_{m}"], color=MODEL_COLORS[m], lw=MODEL_LW[m], label=MODEL_LABELS[m])
    axes[3].set_ylabel("Regret"); axes[3].set_xlabel("Date")
    for ax in axes:
        format_date_axis(ax, year_step); ax.legend(fontsize=8, frameon=False, loc="upper right", ncol=4)
    fig.suptitle(f"Quarterly-PD PTO vs DFL model with Basel IRB allocation ({split_name})", fontsize=12)
    plt.tight_layout(); plt.savefig(fig_dir / "dfl_pd_irb_compact_panel.png", dpi=220, bbox_inches="tight"); plt.show()

    # 8. Predicted vs realised scatter (PD models only - skipped when none selected)
    if PLOT_MODELS_PD:
        fig, axes = plt.subplots(1, len(PLOT_MODELS_PD), figsize=(4 * len(PLOT_MODELS_PD), 4), sharex=True, sharey=True)
        lims = [0.0, max(plot_df["pd_q_realised"].max(), *(plot_df[f"pd_q_hat_{m}"].max() for m in PLOT_MODELS_PD)) * 1.1]
        for ax, m in zip(np.atleast_1d(axes), PLOT_MODELS_PD):
            ax.scatter(plot_df["pd_q_realised"], plot_df[f"pd_q_hat_{m}"], s=18, color=MODEL_COLORS[m], alpha=0.75)
            ax.plot(lims, lims, color="#999999", lw=1, ls=":")
            ax.set_xlim(lims); ax.set_ylim(lims)
            ax.set_title(MODEL_LABELS[m]); ax.set_xlabel("Realised quarterly PD")
            ax.xaxis.set_major_formatter(mticker.PercentFormatter(1.0)); ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
            ax.grid(True, alpha=0.28)
        np.atleast_1d(axes)[0].set_ylabel("Predicted quarterly PD")
        fig.suptitle(f"Predicted versus realised quarterly PD ({split_name}; PD models only)")
        plt.tight_layout(); plt.savefig(fig_dir / "pred_vs_realised_scatter.png", dpi=200, bbox_inches="tight"); plt.show()

    # 9. DFL alpha vs oracle alpha scatter (skipped when no DFL model selected)
    if PLOT_MODELS_DFL:
        fig, axes = plt.subplots(1, len(PLOT_MODELS_DFL), figsize=(4 * len(PLOT_MODELS_DFL), 4), sharex=True, sharey=True)
        for ax, m in zip(np.atleast_1d(axes), PLOT_MODELS_DFL):
            ax.scatter(plot_df["alpha_oracle"], plot_df[f"alpha_hat_{m}"], s=18, color=MODEL_COLORS[m], alpha=0.75)
            ax.plot([0, 1], [0, 1], color="#999999", lw=1, ls=":")
            ax.set_xlim(0, 1); ax.set_ylim(0, 1)
            ax.set_title(MODEL_LABELS[m]); ax.set_xlabel(r"Foresight-oracle $\alpha$")
            ax.grid(True, alpha=0.28)
        np.atleast_1d(axes)[0].set_ylabel(r"DFL $\alpha$")
        fig.suptitle(f"DFL allocation versus foresight oracle ({split_name})")
        plt.tight_layout(); plt.savefig(fig_dir / "dfl_alpha_vs_oracle_scatter.png", dpi=200, bbox_inches="tight"); plt.show()


for split_name, res in results_by_split.items():
    plot_split(res, split_name)


## Learning curves

Averaged across expanding windows within each split; the MLP curve is the first ensemble
seed only.
Early-stopped windows are padded by carrying the last value forward so the per-epoch mean
keeps constant composition. The DFL model gets its own panel: validation REGRET, its
early-stopping criterion (the PD models' panels show validation RMSE and validation loss
in the training space).


In [ ]:
if not training_history.empty:
    for split_name, split_hist in training_history.groupby("split"):
        # Pad each refit's history to the model's max epoch (last value carried forward)
        # so the per-epoch mean has constant composition despite early stopping.
        padded = []
        for model_name, mg in split_hist.groupby("model"):
            max_epoch = int(mg["epoch"].max())
            for _, g in mg.groupby("test_date"):
                gg = (g.sort_values("epoch").set_index("epoch")[["val_rmse_pd_q", "val_loss"]]
                       .reindex(range(1, max_epoch + 1)).ffill().reset_index())
                gg["model"] = model_name
                padded.append(gg)
        mean_history = (pd.concat(padded, ignore_index=True)
                        .groupby(["model", "epoch"], as_index=False)
                        .agg(mean_val_rmse_pd_q=("val_rmse_pd_q", "mean"), mean_val_loss=("val_loss", "mean")))

        fig, axes = plt.subplots(3, 1, figsize=(9, 8.5), sharex=True)
        for model_name, group in mean_history.groupby("model"):
            group = group.sort_values("epoch"); color = MODEL_COLORS.get(model_name, "grey")
            label = MODEL_LABELS.get(model_name, model_name.upper())
            if model_name in DFL_MODELS:
                axes[2].plot(group["epoch"], group["mean_val_loss"], color=color, lw=1.4, label=label)
            elif model_name in ("linear", "mlp"):
                axes[0].plot(group["epoch"], group["mean_val_rmse_pd_q"], color=color, lw=1.4, label=label)
                axes[1].plot(group["epoch"], group["mean_val_loss"], color=color, lw=1.4, label=label)
        axes[0].set_ylabel("Val RMSE PD_Q (PD models)")
        loss_space = "NLL" if LOSS == "nll_vasicek" else f"MSE ({TARGET_TRANSFORM} space)"
        axes[1].set_ylabel(f"Val {loss_space} (PD models)")
        axes[2].set_ylabel("Val regret (DFL model)"); axes[2].set_xlabel("Epoch")
        for ax in axes:
            ax.grid(True, axis="y", lw=0.5, alpha=0.3, linestyle=":"); ax.legend(fontsize=8, frameon=False, loc="upper right")
        fig.suptitle(f"Expanding-window learning curves ({split_name})"); plt.tight_layout()
        fig_dir = FIGURES_DIR / split_name
        fig_dir.mkdir(parents=True, exist_ok=True)
        plt.savefig(fig_dir / "learning_curves.png", dpi=200, bbox_inches="tight"); plt.show()


In [ ]:
BB_RESULTS_PREFIX = "blackbox_quarterly_pd_irb"
## Cross-notebook comparison vs the black box (shared test dates)

# The black-box notebook saves per-date predictions to RESULTS_DIR; if present, compare
# DFL vs black-box regret with the same DM machinery on the shared dates. Run the
# black-box notebook first (or skip - this cell is a no-op if the CSV is absent).
for split_name in EVAL_SPLITS:
    bb_csv = RESULTS_DIR / f"{BB_RESULTS_PREFIX}_predictions_{split_name}.csv"
    if not bb_csv.exists():
        print(f"[{split_name}] {bb_csv.name} not found - run the black-box notebook for this comparison.")
        continue
    bb = pd.read_csv(bb_csv, parse_dates=["date"])
    ours = results_by_split[split_name][["date", "regret_dfl_linear", "regret_dfl_mlp"]]
    merged = ours.merge(bb[["date", "regret_blackbox_linear", "regret_blackbox_mlp"]], on="date", how="inner")
    print(f"=== {split_name}: DFL vs black box on {len(merged)} shared dates ===")
    rows = []
    for a, b in [("regret_dfl_linear", "regret_blackbox_linear"), ("regret_dfl_mlp", "regret_blackbox_mlp")]:
        stat, p = dm_test(merged[a].to_numpy(), merged[b].to_numpy())
        rows.append({"pair": f"{a.replace('regret_', '')} vs {b.replace('regret_', '')}",
                     "mean_regret_dfl": float(merged[a].mean()), "mean_regret_bb": float(merged[b].mean()),
                     "dm_regret": stat, "p_regret": p})
    display(pd.DataFrame(rows).set_index("pair"))